In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from scipy.integrate import solve_ivp
import math
import collections # For defaultdict or similar if needed, but not strictly for this version

# --- 1. Parse the AST into a Python dictionary structure ---
# This is a manual conversion of the provided text AST.
# It aims to capture the structure and semantics for execution.
# Note: Python uses '**' for power, 'and' for logical AND, 'or' for logical OR, '==' for equality.
keymaera_x_ast = {
    "KeYmaeraXSpecification": {
        "DefinitionsBlock": {
            "HP": {
                "controlFunc": {
                    "Body (raw)": "{ { vxo := *; vyo := *; ?vxo^2+vyo^2<=V^2; } ++ { { v := 0; w := 0; } ++ { v := *; ?0<=v; w := *; ?-W<=w & w<=W; r := *; xo := *; yo := *; ?r!=0 & r*w = v; ? abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparable(v) } }",
                    "choices": [
                        # Choice 1: vxo, vyo non-deterministic with constraint
                        {
                            "type": "choice_block",
                            "id": 0, # Internal ID for agent to choose this branch
                            "assignments": [
                                {"variable": "vxo", "value": "*"}, # Agent chooses vxo
                                {"variable": "vyo", "value": "*"}  # Agent chooses vyo
                            ],
                            "tests": ["vxo**2 + vyo**2 <= V**2"] # Constraint on agent's choice
                        },
                        # Choice 2: Nested choices (represented as a list of sub-choices)
                        {
                            "type": "choice_block_nested",
                            "choices": [
                                # Choice 2.1: v=0, w=0 deterministic
                                {
                                    "type": "choice_block",
                                    "id": 1, # Internal ID for agent to choose this branch
                                    "assignments": [
                                        {"variable": "v", "value": "0"},
                                        {"variable": "w", "value": "0"}
                                    ],
                                    "tests": [] # No tests for this branch
                                },
                                # Choice 2.2: v, w, r, xo, yo non-deterministic with constraints
                                {
                                    "type": "choice_block",
                                    "id": 2, # Internal ID for agent to choose this branch
                                    "assignments": [
                                        {"variable": "v", "value": "*"},
                                        {"variable": "w", "value": "*"},
                                        {"variable": "r", "value": "*"},
                                        {"variable": "xo", "value": "*"},
                                        {"variable": "yo", "value": "*"}
                                    ],
                                    "tests": [
                                        "0 <= v",
                                        "-W <= w and w <= W",
                                        "r != 0 and r * w == v",
                                        "abs(x - xo) > admissibleSeparation(v) or abs(y - yo) > admissibleSeparation(v)"
                                    ]
                                }
                            ]
                        }
                    ]
                },
                "stepFunc": {
                    "Body (raw)": "{ x' = v * dx, y' = v * dy, dx' = -w * dy, dy' = w * dx, xo' = vxo, yo' = vyo, t' = 1 ...",
                    "ODEs": [
                        {"variable": "x", "expression": "v * dx"},
                        {"variable": "y", "expression": "v * dy"},
                        {"variable": "dx", "expression": "-w * dy"},
                        {"variable": "dy", "expression": "w * dx"},
                        {"variable": "xo", "expression": "vxo"},
                        {"variable": "yo", "expression": "vyo"},
                        {"variable": "t", "expression": "1"}
                    ],
                    "EvolutionDomain": [
                        "t <= ep",
                        "v >= 0"
                    ]
                }
            },
            "DEF": {
                # Parameters like ep, W, V are assumed to be set externally or fixed for the environment
                "ep": {"type": "Real", "value": None, "parameters": []},
                "W": {"type": "Real", "value": None, "parameters": []},
                "V": {"type": "Real", "value": None, "parameters": []},
                "stopDist": {
                    "type": "Real",
                    "value": "0", # This is a constant function
                    "parameters": ["v"] # But takes v as a parameter
                },
                "accelComp": {
                    "type": "Real",
                    "value": "ep * (v + V)",
                    "parameters": ["v"]
                },
                "admissibleSeparation": {
                    "type": "Real",
                    "value": "stopDist(v) + accelComp(v)",
                    "parameters": ["v"]
                },
                "isWellformedDir": {
                    "type": "Bool",
                    "value": "dx**2 + dy**2 == 1",
                    "parameters": ["dx", "dy"]
                },
                "bounds": {
                    "type": "Bool",
                    "value": "ep > 0 and V >= 0",
                    "parameters": []
                },
                "initialState": {
                    "type": "Bool",
                    "value": "v == 0 and (x-xo)**2 + (y-yo)**2 > 0 and isWellformedDir(dx,dy)",
                    "parameters": ["x", "y", "v", "dx", "dy", "xo", "yo"]
                },
                "assumptions": {
                    "type": "Bool",
                    "value": "bounds() and initialState(x,y,v,dx,dy,xo,yo)",
                    "parameters": ["x", "y", "v", "dx", "dy", "xo", "yo"]
                },
                "loopinv": {
                    "type": "Bool",
                    "value": "v >= 0 and isWellformedDir(dx,dy) and (v > 0 and (abs(x-xo) > stopDist(v) or abs(y-yo) > admissibleSeparation(v)))",
                    "parameters": ["x", "y", "v", "dx", "dy", "xo", "yo"]
                }
            }
        },
        "ProgramVariablesBlock": {
            "VAR": [
                {"name": "x", "type": "Real"},
                {"name": "y", "type": "Real"},
                {"name": "v", "type": "Real"},
                {"name": "a", "type": "Real"}, # 'a' is declared but not used in the provided snippets.
                {"name": "dx", "type": "Real"},
                {"name": "dy", "type": "Real"},
                {"name": "w", "type": "Real"},
                {"name": "r", "type": "Real"},
                {"name": "xo", "type": "Real"},
                {"name": "yo", "type": "Real"},
                {"name": "vxo", "type": "Real"},
                {"name": "vyo", "type": "Real"},
                {"name": "t", "type": "Real"}
            ]
        },
        "PROBLEM": {
            "Precondition (P)": {
                "predicate": "assumptions(x, y, v, dx, dy, xo, yo)"
            },
            "LoopExecution": {
                "control_and_reset": {
                    "statements": [
                        {"type": "call", "function": "controlFunc"},
                        {"type": "assignment", "variable": "t", "value": "0"}
                    ]
                },
                "step_function_with_invariants": {
                    "statement": {"type": "call", "function": "stepFunc"},
                    "invariants": [
                        "t >= 0",
                        "isWellformedDir(dx, dy)",
                        # Note: 'old(variable)' is replaced with 'old_variable' in evaluation scope
                        "(-t*V <= xo - old_xo and xo - old_xo <= t*V)",
                        "(-t*V <= yo - old_yo and yo - old_yo <= t*V)",
                        # Simplified interpretation of (x'=0*dx -> x = old(x))
                        # This means if dx and dy are 0 (no movement), then x and y should not change.
                        "(dx == 0 and dy == 0 and v == 0 and x == old_x)",
                        "(dx == 0 and dy == 0 and v == 0 and y == old_y)",
                        # Simplified interpretation of (x'=v*dx -> ...)
                        # This captures the bounds on displacement based on speed v and time t.
                        "(v != 0 and -t * v <= x - old_x and x - old_x <= t * v)",
                        "(v != 0 and -t * v <= y - old_y and y - old_y <= t * v)"
                    ]
                },
                "Invariant": {
                    "predicate": "loopinv(x, y, v, dx, dy, xo, yo)"
                }
            },
            "Postcondition (Q)": {
                "predicate": "(v > 0 and (x - xo)**2 + (y - yo)**2 > 0)"
            }
        }
    }
}

# --- 2. Define the Gymnasium Environment ---

class KeymaeraXEnv(gym.Env):
    """
    A Gymnasium environment for the KeYmaera X specification.

    This environment simulates a hybrid system based on the provided AST.
    The agent's goal is to navigate the system to satisfy the postcondition
    while maintaining invariants.
    """
    metadata = {"render_modes": ["human"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        self.ast = keymaera_x_ast # Store the parsed AST

        # Define variable names and their initial bounds (can be refined)
        self.var_names = [var["name"] for var in self.ast["KeYmaeraXSpecification"]["ProgramVariablesBlock"]["VAR"]]
        self.state = {name: 0.0 for name in self.var_names} # All variables are Real

        # --- Environment Parameters (from DEF block, set to reasonable defaults) ---
        # These are crucial constants for the environment's physics and logic.
        # In a real scenario, these might be configurable or part of the problem definition.
        self.state['ep'] = 0.1  # Example value for epsilon (time step duration for ODEs)
        self.state['W'] = 1.0   # Example value for max angular velocity (for 'w')
        self.state['V'] = 5.0   # Example value for max speed (for 'v', 'vxo', 'vyo')
        self.state['a'] = 0.0   # 'a' is declared but not used in the provided snippets.

        # Define Observation Space (all state variables)
        # The bounds are important for RL algorithms. These are chosen as reasonable defaults.
        # If not specified in the problem, these are design choices.
        # Note: dx, dy are normalized, so their bounds are [-1, 1].
        low_bounds = {
            'x': -100.0, 'y': -100.0, 'v': 0.0, 'a': -10.0, 'dx': -1.0, 'dy': -1.0,
            'w': -self.state['W'], 'r': -100.0, 'xo': -100.0, 'yo': -100.0,
            'vxo': -self.state['V'], 'vyo': -self.state['V'], 't': 0.0,
            'ep': 0.0, 'W': 0.0, 'V': 0.0 # Parameters are also part of the state
        }
        high_bounds = {
            'x': 100.0, 'y': 100.0, 'v': self.state['V'], 'a': 10.0, 'dx': 1.0, 'dy': 1.0,
            'w': self.state['W'], 'r': 100.0, 'xo': 100.0, 'yo': 100.0,
            'vxo': self.state['V'], 'vyo': self.state['V'], 't': self.state['ep'],
            'ep': 1.0, 'W': 10.0, 'V': 10.0 # Parameters are also part of the state
        }

        # Collect all variable names for observation space in a consistent order
        self.obs_var_names = sorted(list(self.state.keys())) # Sort for consistent order
        self.observation_space = spaces.Box(
            low=np.array([low_bounds.get(var, -np.inf) for var in self.obs_var_names], dtype=np.float32),
            high=np.array([high_bounds.get(var, np.inf) for var in self.obs_var_names], dtype=np.float32),
            shape=(len(self.obs_var_names),),
            dtype=np.float32
        )
        # Create a mapping from variable name to its index in the observation array
        self.obs_var_map = {name: i for i, name in enumerate(self.obs_var_names)}

        # Define Action Space
        # The agent chooses a branch and then values for non-deterministic variables.
        # Branch 0 (id=0): sets vxo, vyo (constrained by V)
        # Branch 1 (id=1): sets v=0, w=0 (no agent input needed for these)
        # Branch 2 (id=2): sets v, w, r, xo, yo (constrained by V, W, etc.)
        # The agent provides values for ALL continuous actions, but only those relevant
        # to the chosen 'choice_id' are used. Others are ignored.
        self.action_space = spaces.Dict({
            "choice_id": spaces.Discrete(3), # 0 for branch 1, 1 for branch 2.1, 2 for branch 2.2
            "vxo_action": spaces.Box(low=-self.state['V'], high=self.state['V'], shape=(1,), dtype=np.float32),
            "vyo_action": spaces.Box(low=-self.state['V'], high=self.state['V'], shape=(1,), dtype=np.float32),
            "v_action": spaces.Box(low=0.0, high=self.state['V'], shape=(1,), dtype=np.float32),
            "w_action": spaces.Box(low=-self.state['W'], high=self.state['W'], shape=(1,), dtype=np.float32),
            "r_action": spaces.Box(low=-10.0, high=10.0, shape=(1,), dtype=np.float32), # r!=0 is a constraint, agent must learn to avoid 0.
            "xo_action": spaces.Box(low=-100.0, high=100.0, shape=(1,), dtype=np.float32),
            "yo_action": spaces.Box(low=-100.0, high=100.0, shape=(1,), dtype=np.float32),
        })

        self.render_mode = render_mode
        self.current_step = 0
        self.max_steps = 100 # Arbitrary maximum steps to prevent infinite loops in an episode

        # Placeholder for the last reward, used in render_frame
        self._current_reward_placeholder = 0.0

    def _get_obs(self):
        """
        Converts the internal state dictionary to a numpy array for observation.
        Defensively converts any (1,)-shaped numpy arrays to scalars.
        """
        obs_list = []
        for var in self.obs_var_names:
            val = self.state[var]
            # If a value is a (1,)-shaped numpy array, extract the scalar.
            # This handles potential subtle state corruptions where a numpy array
            # might have been assigned instead of a plain float.
            if isinstance(val, np.ndarray) and val.shape == (1,):
                obs_list.append(val[0])
            else:
                obs_list.append(val)
        return np.array(obs_list, dtype=np.float32)

    def _get_info(self):
        """Provides additional debugging information."""
        return {
            "current_time": self.state['t'],
            "loop_invariant_satisfied": self._evaluate_predicate(self.ast["KeYmaeraXSpecification"]["PROBLEM"]["LoopExecution"]["Invariant"]["predicate"], current_state=self.state, old_state=self.old_state_for_invariants),
            "precondition_satisfied": self._evaluate_predicate(self.ast["KeYmaeraXSpecification"]["PROBLEM"]["Precondition (P)"]["predicate"], current_state=self.state)
        }

    def _evaluate_predicate(self, predicate_str, current_state=None, old_state=None):
        """
        Evaluates a predicate string within the current environment's state.
        This uses `eval` and is a security risk if `predicate_str` comes from untrusted sources.
        For this example, it's assumed to be from the trusted AST.
        """
        if current_state is None:
            current_state = self.state.copy()

        # Create a local scope for evaluation
        eval_scope = {
            'abs': abs, 'math': math, # Add common math functions
            'np': np, # If any numpy functions are used
        }
        eval_scope.update(current_state) # Add all state variables

        # Handle 'old' variables if old_state is provided
        if old_state:
            for k, v in old_state.items():
                eval_scope[f'old_{k}'] = v

        # Define DEF functions dynamically for evaluation
        def_block = self.ast["KeYmaeraXSpecification"]["DefinitionsBlock"]["DEF"]
        for def_name, def_details in def_block.items():
            if def_details["value"] is not None:
                if def_details["parameters"]:
                    # Create a closure for definitions with parameters
                    def make_def_func(val_str, params, outer_eval_scope):
                        def func(*args):
                            # Create a temporary scope for function call, inheriting from outer_eval_scope
                            func_scope = outer_eval_scope.copy()
                            for i, param_name in enumerate(params):
                                func_scope[param_name] = args[i]
                            try:
                                return eval(val_str, {"__builtins__": None}, func_scope)
                            except Exception as e:
                                # print(f"Error evaluating DEF function {def_name} with value '{val_str}' and params {args}: {e}")
                                return False # Default to False or 0.0 if evaluation fails
                        return func
                    eval_scope[def_name] = make_def_func(def_details["value"], def_details["parameters"], eval_scope)
                else:
                    # For definitions without parameters (like bounds()), evaluate directly
                    try:
                        eval_scope[def_name] = eval(def_details["value"], {"__builtins__": None}, eval_scope)
                    except Exception as e:
                        # print(f"Error evaluating parameterless DEF function {def_name} with value '{def_details['value']}': {e}")
                        eval_scope[def_name] = False # Default to False if evaluation fails

        try:
            return eval(predicate_str, {"__builtins__": None}, eval_scope)
        except Exception as e:
            # print(f"Error evaluating predicate '{predicate_str}': {e}")
            return False # Assume predicate is false if evaluation fails


    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Initialize state variables to default values
        for var_name in self.var_names:
            if var_name not in ['ep', 'W', 'V', 'a']: # Parameters are fixed, not reset by initialState
                self.state[var_name] = 0.0

        # Set initial values for dx, dy to satisfy isWellformedDir (dx^2 + dy^2 = 1)
        # For simplicity, let's start with a fixed valid direction
        self.state['dx'] = 1.0
        self.state['dy'] = 0.0
        # More robust: sample from a unit circle
        # angle = self.np_random.uniform(0, 2 * np.pi)
        # self.state['dx'] = math.cos(angle)
        # self.state['dy'] = math.sin(angle)

        # Apply initialState and assumptions
        # This part often involves sampling within constraints for RL.
        # For now, let's try to find a valid initial state that satisfies:
        # v = 0, (x-xo)^2 + (y-yo)^2 > 0, isWellformedDir(dx,dy)
        self.state['v'] = 0.0
        self.state['x'] = 1.0
        self.state['y'] = 0.0
        self.state['xo'] = 0.0
        self.state['yo'] = 0.0
        self.state['t'] = 0.0
        self.state['vxo'] = 0.0
        self.state['vyo'] = 0.0
        self.state['w'] = 0.0
        self.state['r'] = 0.0

        # Ensure (x-xo)^2 + (y-yo)^2 > 0
        if (self.state['x'] - self.state['xo'])**2 + (self.state['y'] - self.state['yo'])**2 == 0:
            self.state['x'] += 1.0 # Shift x to ensure separation

        # Verify initial state against precondition
        if not self._evaluate_predicate(self.ast["KeYmaeraXSpecification"]["PROBLEM"]["Precondition (P)"]["predicate"]):
            print("Warning: Initial state does not satisfy precondition. This might indicate a problem with initial state setup.")
            # In a real environment, this might lead to an immediate termination or a very negative reward.
            # For demonstration, we'll proceed.

        self.current_step = 0
        # Store initial state for 'old' variables in loop invariants
        self.old_state_for_invariants = self.state.copy()

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info

    def _ode_system(self, t, y_vec, current_env_state):
        """
        Defines the system of ODEs from stepFunc.
        y_vec order must match the order of ODEs in the AST.
        """
        # Map y_vec back to named variables for expression evaluation
        # The order here must match the order in solve_ivp's initial state and return
        ode_vars_order = ['x', 'y', 'dx', 'dy', 'xo', 'yo', 't']
        ode_state = {var: val for var, val in zip(ode_vars_order, y_vec)}

        # Combine with other state variables (v, w, vxo, vyo, ep, W, V, etc.)
        # These are considered constant during a single ODE integration step
        eval_scope = current_env_state.copy()
        eval_scope.update(ode_state)

        # Define DEF functions dynamically for evaluation within ODE system
        def_block = self.ast["KeYmaeraXSpecification"]["DefinitionsBlock"]["DEF"]
        for def_name, def_details in def_block.items():
            if def_details["value"] is not None:
                if def_details["parameters"]:
                    def make_def_func(val_str, params, outer_eval_scope):
                        def func(*args):
                            func_scope = outer_eval_scope.copy()
                            for i, param_name in enumerate(params):
                                func_scope[param_name] = args[i]
                            try:
                                return eval(val_str, {"__builtins__": None}, func_scope)
                            except Exception:
                                return 0.0 # Default to 0 if evaluation fails within ODE
                        return func
                    eval_scope[def_name] = make_def_func(def_details["value"], def_details["parameters"], eval_scope)
                else:
                    try:
                        eval_scope[def_name] = eval(def_details["value"], {"__builtins__": None}, eval_scope)
                    except Exception:
                        eval_scope[def_name] = 0.0

        # Evaluate ODE expressions
        derivatives = []
        for ode in self.ast["KeYmaeraXSpecification"]["DefinitionsBlock"]["HP"]["stepFunc"]["ODEs"]:
            try:
                derivatives.append(eval(ode["expression"], {"__builtins__": None}, eval_scope))
            except Exception as e:
                # print(f"Error evaluating ODE expression '{ode['expression']}': {e}")
                derivatives.append(0.0) # Default to 0 if evaluation fails

        return derivatives

    def step(self, action):
        terminated = False
        truncated = False
        reward = -0.1 # Small negative reward per step to encourage efficiency

        self.current_step += 1
        if self.current_step >= self.max_steps:
            truncated = True

        # Store state before controlFunc for 'old' variables in loop invariants
        self.old_state_for_invariants = self.state.copy()

        # --- 1. Apply controlFunc (discrete action) ---
        choice_id = action["choice_id"]
        control_func_ast = self.ast["KeYmaeraXSpecification"]["DefinitionsBlock"]["HP"]["controlFunc"]
        chosen_branch_details = None

        # Flatten choices for easier access by agent's choice_id
        all_choices_flat = []
        for c in control_func_ast["choices"]:
            if c["type"] == "choice_block":
                all_choices_flat.append(c)
            elif c["type"] == "choice_block_nested":
                all_choices_flat.extend(c["choices"])

        if choice_id < len(all_choices_flat):
            chosen_branch_details = all_choices_flat[choice_id]
        else:
            # Invalid choice_id, penalize and terminate
            reward -= 10.0
            terminated = True
            print(f"Invalid choice_id: {choice_id}. Terminating.")
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # Apply assignments from the chosen branch
        control_success = True
        for assignment in chosen_branch_details["assignments"]:
            var = assignment["variable"]
            val_expr = assignment["value"]

            if val_expr == "*": # Non-deterministic assignment, use agent's action
                # Ensure the action value is within the defined bounds of the action space
                if var == "vxo": self.state[var] = np.clip(action["vxo_action"][0], self.action_space["vxo_action"].low, self.action_space["vxo_action"].high)
                elif var == "vyo": self.state[var] = np.clip(action["vyo_action"][0], self.action_space["vyo_action"].low, self.action_space["vyo_action"].high)
                elif var == "v": self.state[var] = np.clip(action["v_action"][0], self.action_space["v_action"].low, self.action_space["v_action"].high)
                elif var == "w": self.state[var] = np.clip(action["w_action"][0], self.action_space["w_action"].low, self.action_space["w_action"].high)
                elif var == "r": self.state[var] = np.clip(action["r_action"][0], self.action_space["r_action"].low, self.action_space["r_action"].high)
                elif var == "xo": self.state[var] = np.clip(action["xo_action"][0], self.action_space["xo_action"].low, self.action_space["xo_action"].high)
                elif var == "yo": self.state[var] = np.clip(action["yo_action"][0], self.action_space["yo_action"].low, self.action_space["yo_action"].high)
                # else: print(f"Warning: Non-deterministic assignment for unknown variable {var}")
            else: # Deterministic assignment (e.g., v := 0)
                try:
                    self.state[var] = float(val_expr)
                except ValueError:
                    # Should not happen if AST is well-formed for controlFunc assignments
                    pass

        # Evaluate tests (guards) for the chosen branch
        for test_condition in chosen_branch_details["tests"]:
            if not self._evaluate_predicate(test_condition):
                control_success = False
                break

        if not control_success:
            reward -= 5.0 # Penalty for violating controlFunc guards
            # The system cannot proceed as intended. This is a critical failure.
            terminated = True
            # print(f"ControlFunc guards violated. Terminating. Condition: {test_condition}")
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # Reset 't' to 0 after controlFunc, as per PROBLEM block
        self.state['t'] = 0.0

        # --- 2. Execute stepFunc (continuous dynamics) ---
        # Prepare initial state for ODE solver
        ode_vars_order = ['x', 'y', 'dx', 'dy', 'xo', 'yo', 't']
        y0 = [self.state[var] for var in ode_vars_order]

        # Define events for evolution domain (integration stops when t > ep or v < 0)
        def evolution_domain_event_t(t, y_vec, current_env_state):
            # y_vec[6] is 't'
            return current_env_state['ep'] - y_vec[6] # Event occurs when this is 0 (t = ep)
        evolution_domain_event_t.terminal = True
        evolution_domain_event_t.direction = -1 # Event when t crosses ep from below

        def evolution_domain_event_v(t, y_vec, current_env_state):
            # 'v' is not an ODE variable, it's a state variable set by controlFunc.
            # This means 'v' is constant during ODE integration.
            return current_env_state['v'] # Event occurs when v is 0
        evolution_domain_event_v.terminal = True
        evolution_domain_event_v.direction = -1 # Event when v crosses 0 from above (becomes negative)

        # Solve ODEs
        # The integration time span is from t=0 to t=ep (or until an event)
        t_span = (0, self.state['ep'])
        try:
            sol = solve_ivp(
                fun=self._ode_system,
                t_span=t_span,
                y0=y0,
                args=(self.state,), # Pass current environment state (v, w, vxo, vyo, ep, W, V)
                events=[evolution_domain_event_t, evolution_domain_event_v],
                dense_output=True, # To get intermediate states if needed
                max_step=self.state['ep'] / 10.0 # Limit step size for stability
            )

            if sol.status == 0: # Success: integration completed without event
                # Update state with the final values from ODE integration
                final_y = sol.y[:, -1]
                for i, var in enumerate(ode_vars_order):
                    self.state[var] = final_y[i]
            elif sol.status == 1: # Event occurred: integration stopped due to an event
                # Update state with the values at the event
                # sol.y_events is a list of arrays, one for each event type.
                # If multiple events trigger at the same time, take the first one.
                if sol.t_events[0].size > 0: # t=ep event
                    final_y = sol.y_events[0][0]
                elif sol.t_events[1].size > 0: # v=0 event
                    final_y = sol.y_events[1][0]
                else: # Should not happen if status is 1
                    # print("Warning: solve_ivp returned status 1 but no events recorded. Using last step.")
                    final_y = sol.y[:, -1]
                    reward -= 1.0 # Small penalty for unexpected ODE termination
                for i, var in enumerate(ode_vars_order):
                    self.state[var] = final_y[i]
            else: # Integration failed for other reasons (e.g., max_step exceeded, singular matrix)
                # print(f"ODE integration failed with status: {sol.status}, message: {sol.message}")
                reward -= 10.0 # Significant penalty for ODE failure
                terminated = True
                return self._get_obs(), reward, terminated, truncated, self._get_info()

        except Exception as e:
            # print(f"Critical error during ODE integration: {e}")
            reward -= 10.0
            terminated = True
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # --- 3. Check Invariants and Postcondition ---
        # Check Evolution Domain constraints (redundant if events are perfectly set, but good for robustness)
        for constraint in self.ast["KeYmaeraXSpecification"]["DefinitionsBlock"]["HP"]["stepFunc"]["EvolutionDomain"]:
            if not self._evaluate_predicate(constraint):
                # print(f"Evolution Domain violated: {constraint}")
                reward -= 5.0
                terminated = True # Violating evolution domain means the system cannot evolve further
                break
        if terminated:
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # Check Loop Invariant
        loop_inv_satisfied = self._evaluate_predicate(
            self.ast["KeYmaeraXSpecification"]["PROBLEM"]["LoopExecution"]["Invariant"]["predicate"],
            current_state=self.state,
            old_state=self.old_state_for_invariants
        )
        if not loop_inv_satisfied:
            # print("Loop Invariant violated!")
            reward -= 10.0 # Major penalty
            terminated = True # Loop invariant violation implies failure
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # Check Invariants from step_function_with_invariants block
        for inv_str in self.ast["KeYmaeraXSpecification"]["PROBLEM"]["LoopExecution"]["step_function_with_invariants"]["invariants"]:
            if not self._evaluate_predicate(inv_str, current_state=self.state, old_state=self.old_state_for_invariants):
                # print(f"Step function invariant violated: {inv_str}")
                reward -= 7.0 # Significant penalty
                terminated = True
                break
        if terminated:
            return self._get_obs(), reward, terminated, truncated, self._get_info()

        # Check Postcondition (goal)
        postcondition_satisfied = self._evaluate_predicate(
            self.ast["KeYmaeraXSpecification"]["PROBLEM"]["Postcondition (Q)"]["predicate"],
            current_state=self.state
        )
        if postcondition_satisfied:
            reward += 100.0 # Large positive reward for achieving the goal
            terminated = True # Goal achieved, episode ends

        observation = self._get_obs()
        info = self._get_info()

        # Update placeholder for rendering
        self._current_reward_placeholder = reward

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def render(self):
        if self.render_mode == "human":
            return self._render_frame()

    def _render_frame(self):
        # This is a placeholder for actual rendering.
        # For a hybrid system, rendering could involve plotting trajectories,
        # showing current state values, or visualizing the environment.
        # Given the complexity, a simple print of the state is sufficient for demonstration.
        print(f"\n--- Step {self.current_step} ---")
        for k in self.obs_var_names: # Print in consistent order
            v = self.state[k]
            # Ensure value is scalar for printing, in case of accidental numpy array
            if isinstance(v, np.ndarray) and v.shape == (1,):
                v_print = v[0]
            else:
                v_print = v
            print(f"{k}: {v_print:.4f}", end=" | ")
        print(f"\nReward: {self._current_reward_placeholder:.2f}")
        print(f"Loop Inv: {self._evaluate_predicate(self.ast['KeYmaeraXSpecification']['PROBLEM']['LoopExecution']['Invariant']['predicate'], current_state=self.state, old_state=self.old_state_for_invariants)}")
        print("-" * 30)

    def close(self):
        pass # No specific resources to close for this environment


# --- Example Usage ---
if __name__ == "__main__":
    env = KeymaeraXEnv(render_mode="human")
    observation, info = env.reset(seed=42)
    print("Initial Observation (first 5 values):", observation[:5])
    print("Initial Info:", info)

    print("\n--- Running a random agent for 10 steps ---")
    for i in range(10):
        action = env.action_space.sample()
        # Make 'r_action' non-zero for branch 2.2 to satisfy 'r!=0' constraint
        if action["choice_id"] == 2:
            # Ensure r_action is not 0, as r!=0 is a guard.
            # action["r_action"] is a np.ndarray of shape (1,).
            # We modify its single element.
            if action["r_action"][0] == 0:
                action["r_action"][0] = 0.1 if env.np_random.random() > 0.5 else -0.1
            # Ensure v_action is non-negative, as 0<=v is a guard.
            if action["v_action"][0] < 0:
                action["v_action"][0] = 0.0 # Clip to 0 if randomly negative

        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i + 1}: Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")

        if terminated or truncated:
            print("Episode finished.")
            break

    env.close()

    # Test a specific sequence of actions to see if it can satisfy conditions
    print("\n--- Testing a specific sequence of actions ---")
    env = KeymaeraXEnv(render_mode="human")
    observation, info = env.reset(seed=42)

    # Action 1: Agent chooses branch 0 (sets vxo, vyo)
    # Set vxo, vyo to satisfy vxo^2+vyo^2 <= V^2 (V=5, so 1^2+1^2=2 <= 25 is fine)
    action1 = {
        "choice_id": 0,
        "vxo_action": np.array([1.0], dtype=np.float32),
        "vyo_action": np.array([1.0], dtype=np.float32),
        "v_action": np.array([0.0], dtype=np.float32), # Ignored for choice_id 0
        "w_action": np.array([0.0], dtype=np.float32), # Ignored
        "r_action": np.array([1.0], dtype=np.float32), # Ignored
        "xo_action": np.array([0.0], dtype=np.float32), # Ignored
        "yo_action": np.array([0.0], dtype=np.float32), # Ignored
    }
    obs, reward, terminated, truncated, info = env.step(action1)
    print(f"Step 1 (controlFunc branch 0): Reward={reward:.2f}, Terminated={terminated}")
    if terminated or truncated:
        print("Episode finished early.")
    else:
        # Action 2: Agent chooses branch 2 (sets v, w, r, xo, yo)
        # Need to satisfy: 0<=v, -W<=w<=W, r!=0 & r*w=v, abs(x-xo)>admissibleSeparation(v) | abs(y-yo)>admissibleSeparation(v)
        # Let's try to set v=1, w=0.1, r=10 (so r*w = 1.0 = v)
        # This will make the agent move.
        action2 = {
            "choice_id": 2,
            "vxo_action": np.array([0.0], dtype=np.float32), # Ignored for choice_id 2
            "vyo_action": np.array([0.0], dtype=np.float32), # Ignored
            "v_action": np.array([1.0], dtype=np.float32), # v=1
            "w_action": np.array([0.1], dtype=np.float32), # w=0.1 (within W=1)
            "r_action": np.array([10.0], dtype=np.float32), # r=10 (r*w = 1.0 = v)
            "xo_action": np.array([0.0], dtype=np.float32), # Agent's choice for xo,yo
            "yo_action": np.array([0.0], dtype=np.float32), # Agent's choice for xo,yo
        }
        obs, reward, terminated, truncated, info = env.step(action2)
        print(f"Step 2 (controlFunc branch 2): Reward={reward:.2f}, Terminated={terminated}")
        if terminated or truncated:
            print("Episode finished early.")
        else:
            # Action 3: Agent chooses branch 1 (sets v=0, w=0)
            # This is a 'safe' action to stop the agent's movement.
            action3 = {
                "choice_id": 1,
                "vxo_action": np.array([0.0], dtype=np.float32), # Ignored for choice_id 1
                "vyo_action": np.array([0.0], dtype=np.float32), # Ignored
                "v_action": np.array([0.0], dtype=np.float32), # Ignored
                "w_action": np.array([0.0], dtype=np.float32), # Ignored
                "r_action": np.array([1.0], dtype=np.float32), # Ignored
                "xo_action": np.array([0.0], dtype=np.float32), # Ignored
                "yo_action": np.array([0.0], dtype=np.float32), # Ignored
            }
            obs, reward, terminated, truncated, info = env.step(action3)
            print(f"Step 3 (controlFunc branch 1): Reward={reward:.2f}, Terminated={terminated}")

    env.close()


--- Step 0 ---
V: 5.0000 | W: 1.0000 | a: 0.0000 | dx: 1.0000 | dy: 0.0000 | ep: 0.1000 | r: 0.0000 | t: 0.0000 | v: 0.0000 | vxo: 0.0000 | vyo: 0.0000 | w: 0.0000 | x: 1.0000 | xo: 0.0000 | y: 0.0000 | yo: 0.0000 | 
Reward: 0.00
Loop Inv: False
------------------------------
Initial Observation (first 5 values): [5. 1. 0. 1. 0.]
Initial Info: {'current_time': 0.0, 'loop_invariant_satisfied': False, 'precondition_satisfied': False}

--- Running a random agent for 10 steps ---
Step 1: Reward=-10.10, Terminated=True, Truncated=False
Episode finished.

--- Testing a specific sequence of actions ---

--- Step 0 ---
V: 5.0000 | W: 1.0000 | a: 0.0000 | dx: 1.0000 | dy: 0.0000 | ep: 0.1000 | r: 0.0000 | t: 0.0000 | v: 0.0000 | vxo: 0.0000 | vyo: 0.0000 | w: 0.0000 | x: 1.0000 | xo: 0.0000 | y: 0.0000 | yo: 0.0000 | 
Reward: 0.00
Loop Inv: False
------------------------------
Step 1 (controlFunc branch 0): Reward=-10.10, Terminated=True
Episode finished early.
